In [ ]:
# ============================================================
# 출시 후 분석 게임별 결과 통합
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

POSTLAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "postlaunch"
RUNS_DIR = POSTLAUNCH_OUTPUT_DIR / "runs"

MASTER_DIR = POSTLAUNCH_OUTPUT_DIR / "master"
MASTER_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PREPROCESS_DIR = MASTER_DIR / "postlaunch_preprocess_data"
MASTER_PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)


def read_run_csv(run_dir, relative_path):
    path = run_dir / relative_path

    if not path.exists():
        return None

    df = pd.read_csv(path, dtype={"recommendationid": "string"})
    df["source_run"] = run_dir.name
    return df


def concat_run_csv(relative_path, drop_duplicates_subset=None):
    dfs = []

    for run_dir in sorted(RUNS_DIR.iterdir()):
        if not run_dir.is_dir():
            continue

        df = read_run_csv(run_dir, relative_path)

        if df is not None and len(df) > 0:
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    merged = pd.concat(dfs, ignore_index=True)

    if drop_duplicates_subset:
        valid_subset = [col for col in drop_duplicates_subset if col in merged.columns]
        if valid_subset:
            merged = merged.drop_duplicates(subset=valid_subset, keep="last")

    return merged


# ============================================================
# 04번 LLM 원천 산출물 통합
# ============================================================

llm_input_reviews = concat_run_csv(
    "llm_input_reviews.csv",
    drop_duplicates_subset=["recommendationid"],
)

llm_filter_log = concat_run_csv(
    "llm_input_filter_log.csv",
)

llm_sample_summary = concat_run_csv(
    "llm_input_sample_summary.csv",
)

llm_review_result = concat_run_csv(
    "llm_review_analysis_result.csv",
    drop_duplicates_subset=["recommendationid"],
)

llm_issue_tags_flat = concat_run_csv(
    "llm_issue_tags_flat.csv",
    drop_duplicates_subset=["appid", "recommendationid", "llm_issue_category", "llm_issue_evidence"],
)

llm_input_reviews.to_csv(MASTER_DIR / "llm_input_reviews.csv", index=False, encoding="utf-8-sig")
llm_filter_log.to_csv(MASTER_DIR / "llm_input_filter_log.csv", index=False, encoding="utf-8-sig")
llm_sample_summary.to_csv(MASTER_DIR / "llm_input_sample_summary.csv", index=False, encoding="utf-8-sig")
llm_review_result.to_csv(MASTER_DIR / "llm_review_analysis_result.csv", index=False, encoding="utf-8-sig")
llm_issue_tags_flat.to_csv(MASTER_DIR / "llm_issue_tags_flat.csv", index=False, encoding="utf-8-sig")


# ============================================================
# 04-1번 게임별 전처리 산출물 통합
# ============================================================

postlaunch_review_base = concat_run_csv(
    "postlaunch_preprocess_data/postlaunch_review_base.csv",
    drop_duplicates_subset=["recommendationid"],
)

postlaunch_issue_summary = concat_run_csv(
    "postlaunch_preprocess_data/postlaunch_issue_summary.csv",
    drop_duplicates_subset=["appid", "llm_issue_category"],
)

postlaunch_patch_ops_evidence_base = concat_run_csv(
    "postlaunch_preprocess_data/postlaunch_patch_ops_evidence_base.csv",
    drop_duplicates_subset=["appid", "llm_issue_category"],
)

tableau_postlaunch_source = concat_run_csv(
    "postlaunch_preprocess_data/tableau_postlaunch_patch_ops_source.csv",
    drop_duplicates_subset=["appid", "recommendationid", "llm_issue_category"],
)

postlaunch_review_base.to_csv(
    MASTER_PREPROCESS_DIR / "postlaunch_review_base.csv",
    index=False,
    encoding="utf-8-sig",
)

postlaunch_issue_summary.to_csv(
    MASTER_PREPROCESS_DIR / "postlaunch_issue_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

postlaunch_patch_ops_evidence_base.to_csv(
    MASTER_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv",
    index=False,
    encoding="utf-8-sig",
)

tableau_postlaunch_source.to_csv(
    MASTER_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 통합 결과 확인
# ============================================================

check_rows = [
    {
        "파일": "llm_input_reviews.csv",
        "행 수": len(llm_input_reviews),
        "저장 위치": str(MASTER_DIR / "llm_input_reviews.csv"),
    },
    {
        "파일": "llm_review_analysis_result.csv",
        "행 수": len(llm_review_result),
        "저장 위치": str(MASTER_DIR / "llm_review_analysis_result.csv"),
    },
    {
        "파일": "llm_issue_tags_flat.csv",
        "행 수": len(llm_issue_tags_flat),
        "저장 위치": str(MASTER_DIR / "llm_issue_tags_flat.csv"),
    },
    {
        "파일": "postlaunch_review_base.csv",
        "행 수": len(postlaunch_review_base),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "postlaunch_review_base.csv"),
    },
    {
        "파일": "postlaunch_issue_summary.csv",
        "행 수": len(postlaunch_issue_summary),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "postlaunch_issue_summary.csv"),
    },
    {
        "파일": "postlaunch_patch_ops_evidence_base.csv",
        "행 수": len(postlaunch_patch_ops_evidence_base),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"),
    },
    {
        "파일": "tableau_postlaunch_patch_ops_source.csv",
        "행 수": len(tableau_postlaunch_source),
        "저장 위치": str(MASTER_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"),
    },
]

check_df = pd.DataFrame(check_rows)
display(check_df)

print("통합 완료")
print("통합된 게임 수:", postlaunch_review_base["appid"].nunique() if "appid" in postlaunch_review_base.columns else 0)

if "game_name" in postlaunch_review_base.columns:
    display(
        postlaunch_review_base
        .groupby(["appid", "game_name"], dropna=False)
        .agg(review_count=("recommendationid", "nunique"))
        .reset_index()
        .sort_values("review_count", ascending=False)
    )